In [16]:
from pathlib import Path
from collections import Counter
from datetime import datetime, time
import pandas as pd

DATA_DIR = Path(Path.cwd())
COUNTRY_FILES = {"US": DATA_DIR / "data/USvideos.csv", "GB": DATA_DIR / "data/GBvideos.csv", "IN": DATA_DIR / "data/INvideos.csv"}
CATEGORY_FILE = DATA_DIR / "data/categories.csv"

VIRAL_VIEW_THRESHOLD = 1_000_000
STEADY_VIEW_THRESHOLD = 100_000
DIVISIVE_LIKE_RATIO_THRESHOLD = 0.70
STEADY_LIKE_RATIO_THRESHOLD = 0.85
MIN_RATINGS_FOR_RATIO = 100

run_started_at = datetime.now().astimezone()
print(f"Run started at {run_started_at:%Y-%m-%d %H:%M:%S %Z}")
print(f"Countries {', '.join(COUNTRY_FILES)}")
print(f"Data folder: {DATA_DIR.resolve()}")
print(Path.cwd())

Run started at 2026-10-29 18:51:24 India Standard Time
Countries US, GB, IN
Data folder: D:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations
d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations


In [17]:
REQUIRED_COLUMNS = {
    "video_id", "trending_date", "title", "channel_title", "category_id",
    "publish_time", "views", "likes", "dislikes", "comment_count",
    "comments_disabled", "ratings_disabled", "video_error_or_removed"
}

def load_country(path, country, max_rows=None):
    df = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS.difference(df.columns)
    if missing_columns:
        raise ValueError(f"some required columns are missing: {missing_columns}")
    df['country'] = country
    records = df.to_dict('records')
    print(f"loaded from {country} {len(records)} records in path {Path(path).name}")
    return records

all_records = []
records_by_country = {}
for country, path in COUNTRY_FILES.items():
    print(path)
    country_records = load_country(path, country)  
    records_by_country[country] = country_records
    all_records.extend(country_records)
print(f"total records is {len(all_records):,}")    
print(all_records[1:5])

d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations\data\USvideos.csv
loaded from US 16080 records in path USvideos.csv
d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations\data\GBvideos.csv
loaded from GB 15075 records in path GBvideos.csv
d:\Python\AIML\Practice\module_2\trendpulse\part1_python_foundations\data\INvideos.csv
loaded from IN 15075 records in path INvideos.csv
total records is 46,230
[{'video_id': 'Ugyql2NlPAd', 'trending_date': '17.14.11', 'title': 'Surprising Cricket Review', 'channel_title': 'IncredibleChallenge41', 'category_id': 25, 'publish_time': '2017-11-12T22:35:48.000Z', 'tags': 'Winter', 'views': 198385, 'likes': 1159.0, 'dislikes': 492, 'comment_count': 1446.0, 'thumbnail_link': 'https://i.ytimg.com/vi/Ugyql2NlPAd/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about bangalore. Business enquiries in the channel ab

In [18]:
category_df = pd.read_csv(CATEGORY_FILE)
category_records = category_df.to_dict('records')

category_names = {}
for row in category_records:
    try:
        category_names[int(row['category_id'])] = str(row['category_name'])
    except (ValueError, TypeError, KeyError):
        continue  

print(f"total category records: {len(category_names)}")
print(f"Example lookup: {category_names.get(10, "Unknown category")}")    

total category records: 17
Example lookup: Music


In [19]:
NUMERIC_FIELDS = ["views", "likes", "dislikes", "comment_count"]
BOOLEAN_FIELDS = ["comments_disabled", "ratings_disabled", "video_error_or_removed"]
PARSE_FAILURES = {field: 0 for field in NUMERIC_FIELDS + BOOLEAN_FIELDS + ["trending_date", "publish_time"]}

def to_nonnegative_int(value, field_name):
    try:    
        if value is None or (isinstance(value, float) and value != value) or str(value).strip() == "":
            raise ValueError("Blank numeric value")
        number = int(value)
        if number < 0:
            raise ValueError("Number is negative")
        return number
    except(TypeError, ValueError, OverflowError):
        PARSE_FAILURES[field_name] += 1
        return 0

def to_bool(value, field_name):
    if isinstance(value, bool):
        return value
    if value is None or (isinstance(value, float) and value != value):
        PARSE_FAILURES[field_name] += 1
        return False
    normalized = str(value).strip().lower()
    print(normalized,"norm")
    if normalized in {"true": "1", "yes": "y"}:   
        return True
    if normalized in {"false": "0", "no": "n"}:
        return False
    PARSE_FAILURES[field_name] += 1
    return False

def parse_trending_date(value):
    try:
        return datetime.strptime(str(value).strip(), "%y.%d.%m").date()
    except (ValueError, TypeError):
        PARSE_FAILURES["trending_time"] += 1
        return None
    
def parse_publish_time(value):
    try:
        raw = str(value).strip()
        if raw.endswith("Z"):
            raw = raw[:-1]
        return datetime.fromisoformat(raw)
    except (ValueError, TypeError):
            PARSE_FAILURES["publish_time"] += 1
            return None

cleaned_records = []
for raw_record in all_records:
    record = dict(raw_record)
    for field in NUMERIC_FIELDS:
        record[field] = to_nonnegative_int(record.get(field), field)
    for field in BOOLEAN_FIELDS:
         record[field] = to_bool(record.get(field),field)    
    record["trending_date"] = parse_trending_date(record.get("trending_date"))   
    record["publish_time"] = parse_publish_time(record.get("publish_time")) 
    try:
        record["category_id"] = int(record["category_id"])
    except (ValueError, TypeError, KeyError):
        record["category_id"] = None

    try:
        record["category_name"] = category_names.get(record['category_id'], "Unknown category")
    except (ValueError, TypeError, KeyError):
        record["category_name"] = None
    cleaned_records.append(record)

print(f"records length: {len(cleaned_records)}")
print("Parse-failure counts (includes blanks and invalid values):")
for field, count in PARSE_FAILURES.items():
    print(f"{field}, {count}")

records length: 46230
Parse-failure counts (includes blanks and invalid values):
views, 0
likes, 12
dislikes, 0
comment_count, 36
comments_disabled, 0
ratings_disabled, 0
video_error_or_removed, 0
trending_date, 0
publish_time, 0


In [20]:
video_snapshots = []
for record in cleaned_records:
    video_snapshots.append((record["video_id"], record["trending_date"], record["views"], record["likes"]))

print("Example tuples:")
print(video_snapshots[0])
print(f"Tuple immutable: {isinstance(video_snapshots[0], tuple)}")  

video_ids_by_country = {}

for country, records in records_by_country.items():
    ids = set()
    for record in records:
        video_id = record.get("video_id")
        if video_id:
            ids.add(str(video_id))
    video_ids_by_country[country] = ids
    print(f"{country}: {len(ids):,} unique video IDs")
us_gb_shared = video_ids_by_country["US"] & video_ids_by_country["GB"]
us_only = video_ids_by_country["US"] - video_ids_by_country["GB"]
gb_only = video_ids_by_country["GB"] & video_ids_by_country["US"]

print(f"Shared ids: {us_gb_shared}")
print(f"US only: {us_only}")
print(f"GB only: {gb_only}")

channel_counts = Counter()

for record in cleaned_records:
    channel = record["channel_title"]
    if channel:
        channel_counts[channel] += 1

top_15_channel = channel_counts.most_common(15)
print(top_15_channel)
print("\nTop 15 channels by trending appearances:")
for rank, (channel, appearance) in enumerate(top_15_channel, start=1):
    print(f"{rank} {channel} {appearance}")   

country_category_counts = {}
for record in cleaned_records:
    country = record["country"]
    category = record["category_name"]
    if country not in country_category_counts:
        country_category_counts[country] = {}
    country_category_counts[country][category] =  country_category_counts[country].get(category, 0) + 1

print("\nNested summary (first five categories per country):")
for country, category_counts in country_category_counts.items():
    print(f"{country}:")
    for category, count in list(category_counts.items())[:5]:
        print(f"  {category}: {count:,}")        

Example tuples:
('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
Tuple immutable: True
US: 5,068 unique video IDs
GB: 4,113 unique video IDs
IN: 5,878 unique video IDs
Shared ids: {'Po97ljwmeZm', '64HMUuGDZKC', 'zUgsdK9-dhK', 'SSXHMd_9WaB', 'N7JLoKQ_gkb', '3qCkIrks52Y', 'YgljIOh3tQW', 'oDUG7GXDV0-', 'rOBLnf4BxsQ', 'JiKQ4c3Hf_D', 'zs6SHJWtCcY', 'o6xpVPO45x9', '_RWm595C7an', '1dfSqoP6AWv', 'bJw6cRyrTGe', 'bTyxGD-J04-', 'TD9LzgJ-aa4', 'oO6V15qeZYm', 'HizrcajVcGb', '0B3bCL2UO2y', 'wHS7m9Hi7wN', 'hrhxMAMZcBM', 'uBdTiia9dVO', 'xhw0CADq1Q8', 'cvXq3xagZz-', 'khsru25Sj82', 'GXOc3K6oSbk', 'jSnY35duGb5', 'KDAc90So7lp', 'pTgfieQr8gD', 'rhFPsiKfVyA', 'bXnYEodNkC8', 'IVr9NXeXvTZ', 'CP5BIJEQmxE', '5_vLwTVq3s-', 'wLqyIlKijps', 'hyu9CtEUvjA', '-YVDnP9rR2D', '057IFL637-N', 'ENRe3Bbi38W', 'GPEqd807han', 'RjwPneuRQiR', '2i0BT5fDj1C', 'MNWCrB16RFP', 'WP_Q7jkQVyB', 'OGc-p2hoHHM', 'OKyfUtpycz1', 'A-U6P8eTT7f', 'ycJSQdzzjJH', '6wb5p2ToWfz', 'Ugyql2NlPAd', '_gHvOU3z7un', '_sQkkW5b-t_', '15IFX6aqs1b', 

In [21]:
def title_features(title):
    """Return simple title features using core Python only."""
    if title is None:
        title = ""
    title = str(title)
    words = title.split()
    uppercase_words = 0
    for word in words:
        letters = [char for char in word if char.isalpha()]
        if letters and all(char.isupper() for char in letters):
            uppercase_words += 1

    exclamation_marks = 0
    question_marks = 0
    for char in title:
        if char == "!":
            exclamation_marks += 1
        elif char == "?":
            question_marks += 1

    # range() with index access, as required by the task.
    first_words = []
    for index in range(min(3, len(words))):
        first_words.append(words[index])

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "word_count": len(words),
        "first_three_words": first_words
    }

print(title_features("NEW Amazing VIDEO!!! Is This Real?"))

{'uppercase_words': 2, 'exclamation_marks': 3, 'question_marks': 1, 'word_count': 6, 'first_three_words': ['NEW', 'Amazing', 'VIDEO!!!']}


In [22]:
# while loop: stop at the first valid record over the view threshold.
# continue skips records without a usable video ID; break ends the search once a match is found.
index = 0
first_high_view_record = None
while index < len(cleaned_records):
    candidate = cleaned_records[index]
    index += 1
    if not candidate.get("video_id"):
        continue
    if candidate["views"] > VIRAL_VIEW_THRESHOLD:
        first_high_view_record = candidate
        break

if first_high_view_record:
    print(
        "First video over threshold:",
        first_high_view_record["title"],
        "| views =", f'{first_high_view_record["views"]:,}'
    )
else:
    print("No video found above the configured view threshold.")

First video over threshold: Official Cats Speech EXCLUSIVE | views = 2,841,278


In [23]:
def classify_video(record):
    """Assign one transparent rule-based label to a cleaned video record."""
    views = record.get("views", 0)
    likes = record.get("likes", 0)
    dislikes = record.get("dislikes", 0)
    ratings_disabled = record.get("ratings_disabled", False)
    total_ratings = likes + dislikes

    # Nested conditions handle missing/disabled rating edge cases first.
    if ratings_disabled:
        label = "Unrated"
    else:
        if total_ratings == 0:
            label = "Unrated"
        else:
            like_ratio = likes / total_ratings
            if views >= VIRAL_VIEW_THRESHOLD:
                label = "Viral Hit"
            elif total_ratings >= MIN_RATINGS_FOR_RATIO and like_ratio < DIVISIVE_LIKE_RATIO_THRESHOLD:
                label = "Divisive"
            elif views >= STEADY_VIEW_THRESHOLD and like_ratio >= STEADY_LIKE_RATIO_THRESHOLD:
                label = "Steady Performer"
            else:
                label = "Low Engagement"
    return label

# Hand-trace example:
example_record = {
    "video_id": "example12345",
    "views": 250_000,
    "likes": 120,
    "dislikes": 80,
    "ratings_disabled": False,
    "comments_disabled": False
}
example_label = classify_video(example_record)
print("Hand-trace example label:", example_label)
print("Trace: ratings are enabled; total ratings = 200; like ratio = 120/200 = 0.60;")
print("views are below 1,000,000; ratings >= 100 and ratio < 0.70, so label = Divisive.")
assert example_label == "Divisive"

Hand-trace example label: Divisive
Trace: ratings are enabled; total ratings = 200; like ratio = 120/200 = 0.60;
views are below 1,000,000; ratings >= 100 and ratio < 0.70, so label = Divisive.


In [24]:
label_counts = Counter()
for record in cleaned_records:
    record["trend_label"] = classify_video(record)
    label_counts[record["trend_label"]] += 1

print("Records by classification label:")
for label, count in sorted(label_counts.items()):
    print(f"{label}: {count:,}")

Records by classification label:
Divisive: 809
Low Engagement: 9,686
Steady Performer: 25,599
Unrated: 544
Viral Hit: 9,592


In [25]:
def nonempty_text(record, field_name):
    """Return a stripped text value, or an empty string for missing/blank data."""
    value = record.get(field_name)
    if value is None:
        return ""
    return str(value).strip()

# Use the refactored helper consistently.
channel_counts_refactored = Counter()
for record in cleaned_records:
    channel = nonempty_text(record, "channel_title")
    if not channel:
        continue
    channel_counts_refactored[channel] += 1

print("Top 5 channels using refactored helper:")
for channel, count in channel_counts_refactored.most_common(5):
    print(f"{channel}: {count:,}")

Top 5 channels using refactored helper:
QuietChallenge75: 2,464
OfficialReview41: 1,586
UltimateUnboxing65: 1,433
InsaneReaction46: 1,233
Brand NewChallenge45: 1,175


In [26]:
assert len(cleaned_records) >= 40_000
assert len(records_by_country) >= 3
assert all(isinstance(record, dict) for record in cleaned_records)
assert all(isinstance(item, tuple) and len(item) == 4 for item in video_snapshots)
assert isinstance(category_names, dict)
assert all("trend_label" in record for record in cleaned_records)

print("ACCEPTANCE CHECKS PASSED")
print(f"Countries loaded: {len(records_by_country)}")
print(f"Total rows: {len(cleaned_records):,}")
print(f"Unique category IDs in lookup: {len(category_names)}")
print(f"Labels generated: {sum(label_counts.values()):,}")
print("Notebook completed at:", datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z"))

ACCEPTANCE CHECKS PASSED
Countries loaded: 3
Total rows: 46,230
Unique category IDs in lookup: 17
Labels generated: 46,230
Notebook completed at: 2026-10-29 18:51:25 India Standard Time
